### ***RAG Application***
#### **End-to-End Rag Application**

In [48]:
### load the Environment Variables

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [49]:
### Check/Validate the path to load the files from the Document
import os

path = "../kubernetes"

if os.path.exists(path):
    print("Path is valid")
else:
    print("Path is not valid")

Path is valid


In [50]:
#load the documents using DirectoryLoader 

from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()

print("Number Of Documents:",len(documents))

Number Of Documents: 3983


In [51]:
### create a chunks using splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [52]:
###BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k=10

In [53]:
####Initalize the embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model = "BAAI/bge-large-en-v1.5",
    model_kwargs = {"device":"cpu"},
    encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [54]:
### load the vcctors from Chroma DB
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)


In [55]:
#### retriever for similarity search.
vector_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":10}
)

In [56]:
### Hybrid Search
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights = [0.7,0.3]
)

In [57]:
### Lets test with hybrid retriever and how many candidates are retrieved
docs = hybrid_retriever.invoke("What is Kubernetes Deployment?")
print(len(docs))

20


In [58]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [59]:
### Get top 5 final document after reranking

def retrieve_and_rerank(query, k=5):

    retrieved_docs = hybrid_retriever.invoke(query)


    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ##zip the scores and retrieved order the documents based on score from highest to lowest

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    ## Get top k documents using ranked_docs and k value
    top_docs = [doc for (doc,score) in ranked_docs[:k]]

    return top_docs

In [60]:
### Format without unnecessary context

def build_context(documents):

    context = ""

    for i,doc in enumerate(documents,start=1):
        source = doc.metadata.get("source")
        source = source.replace("\\","/")
        source = source.split("/")[-1]
        #print(source)
        context+=f"""
        
    Source {source}
    Page: {doc.metadata.get("page")}
    Content: {doc.page_content}
        """
    return context



In [61]:
### Design a prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the context does not contain the answer, say:
   "I don't have information based on the provided documents."
4. Keep the answer concise and directly relevant.
5. List the sources and page numbers used at the very end of your response.
6. Format the sources strictly like this:
[Source: filename, Page: number]

Context:
{context}

Question:
{question}

Answer:
""")

In [62]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-safeguard-20b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027B0BD32750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027B0BD185C0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

#### ***Step 1: Deduplicate the reranked results before building the LLM context.***

##### **In Deduplication what if page/Documents splits into three chunks, these chunks points to the same page right but with different content may be it's useful we have to handle it otherwise it may lead to hallucination.**

In [63]:
query = "What is Kubernetes Deployment?"

top_5_chunks = retrieve_and_rerank(query)

for i,chunk in enumerate(top_5_chunks,start=1):
    print(f"------ Document {i} --------")
    print("Page:",chunk.metadata.get("page"))

------ Document 1 --------
Page: 9
------ Document 2 --------
Page: 5
------ Document 3 --------
Page: 638
------ Document 4 --------
Page: 6
------ Document 5 --------
Page: 12


In [64]:
queries = [
    "What is a Kubernetes Pod and what resources do containers in a Pod share?",
    "How do containers within the same Pod communicate and share storage?",
    "What happens when a Deployment creates multiple replicas and how are those Pods maintained?",
    "How does a Deployment manage ReplicaSets and Pods during an update?",
    "How does a ReplicaSet maintain the desired number of Pods?",
    "How are Deployments, ReplicaSets, and Pods related to each other?",
    "How does a Kubernetes Service select Pods and provide network access to them?",
    "How does a Service use label selectors, clusterIP, and endpoints to route traffic to Pods?",
    "What happens when a Pod managed by a ReplicaSet is deleted?",
    "How does Kubernetes reconcile the desired state with the actual state?",
    "What happens when a Deployment Pod fails or is deleted?",
    "How can multiple containers in a Pod share networking and storage?",
    "How does a Deployment perform rolling updates using ReplicaSets?",
    "How can a Service provide access to Pods created by a Deployment?",
    "What information can a ConfigMap provide to Pods and how can Pods consume it?"
]
for query in queries:
    print("Query:",query)
    top_5_chunks = retrieve_and_rerank(query)

    for i,chunk in enumerate(top_5_chunks,start=1):
        print(f"------ Document {i} --------")
        print("Page:",chunk.metadata.get("page"))
        print("Chunk ID:",chunk.id)
        print("Page Content:",chunk.page_content)
    print("#"*70)

Query: What is a Kubernetes Pod and what resources do containers in a Pod share?
------ Document 1 --------
Page: 12
Chunk ID: 906f7485-b408-41c8-bcf2-a7952ea74fd3
Page Content: Learn about Kubernetes Nodes.
Troubleshoot deployed applications.
Kubernetes Pods
When you created a Deployment in Module 2, Kubernetes created a Pod to host your
application instance. A Pod is a Kubernetes abstraction that represents a group of one or more
application containers (such as Docker), and some shared resources for those containers. Those
resources include:
Shared storage, as Volumes
Networking, as a unique cluster IP address
Information about how to run each container, such as the container image version or
specific ports to use
A Pod models an application-specific "logical host" and can contain different application
------ Document 2 --------
Page: 90
Chunk ID: None
Page Content: cohesive unit of service. The containers in a Pod are automatically co-located and co-scheduled
on the same physical or

In [65]:
from langchain_core.tracers import LangChainTracer

custom_tracer = LangChainTracer(project_name="RAG Optimization For Kubernetes Docs")

In [66]:
config = {"run_name":"Rag Optimization","callbacks":[custom_tracer]}

In [67]:
def final_rag_response(query):

    top_5_docs = retrieve_and_rerank(query)
    context = build_context(top_5_docs)

    messages = prompt.invoke({"question":query,"context":context})

    response = llm.invoke(messages,config=config).content

    return response

In [68]:
queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?",
    "How does a Deployment manage ReplicaSets and Pods during an update?",
    "How does a Service use label selectors, clusterIP, and endpoints to route traffic to Pods?",
]

for query in queries:
    print("Query:",query)
    response = final_rag_response(query)
    print("Response:",response)
    print("#"*60)

Query: What is a Kubernetes Deployment?
Response: A Kubernetes Deployment is an object that defines the desired state for a set of application Pods—specifying how many replicas should run, what image to use, and other pod settings. The Deployment controller continuously watches those Pods, automatically creating, scaling, updating, and replacing instances to keep the actual state in sync with the desired state and to provide self‑healing when nodes or containers fail.  

[Source: Concepts.pdf, Page: 5]  
[Source: Tutorials.pdf, Page: 9]  
[Source: Tutorials.pdf, Page: 2]
############################################################
Query: What is a Kubernetes Pod?
Response: A Kubernetes Pod is the smallest deployable unit in the Kubernetes object model. It is a logical host that contains one or more containers (typically a single container in most cases) that share the same network namespace, storage volumes, and other resources, and are scheduled to run together on the same node. Pods 

In [69]:
queries = ["How does a Deployment manage ReplicaSets and Pods during an update?"]

for query in queries:
    print("Query:",query)
    response = final_rag_response(query)
    print("Response:",response)
    print("#"*60)

Query: How does a Deployment manage ReplicaSets and Pods during an update?


Response: A Deployment updates its Pods by creating a new ReplicaSet from the updated PodTemplateSpec and then gradually shifting traffic from the old ReplicaSet to the new one. The Deployment controller controls the rate of this rollout, scaling down the old ReplicaSet while scaling up the new one, and updates the Deployment’s revision to reflect the change.  

[Source: Concepts.pdf, Page: 131]  
[Source: Concepts.pdf, Page: 130]
############################################################


#### ***test citation correctness across multiple queries***

In [70]:
queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?",
    "How does a Deployment manage ReplicaSets and Pods during an update?",
    "How does a Service use label selectors, clusterIP, and endpoints to route traffic to Pods?",
    "What happens when a Pod managed by a ReplicaSet is deleted?",
    "How does Kubernetes reconcile the desired state with the actual state?",
    "How do containers within the same Pod communicate and share storage?",
]
for query in queries:
    print("Query:",query)
    response = final_rag_response(query)
    print("Response:",response)
    print("#"*60)

Query: What is a Kubernetes Deployment?


Response: A Kubernetes Deployment is an object that specifies the desired state of an application—such as the number of pod replicas—and lets the control plane create, update, and manage those pods. The Deployment controller continuously monitors the pods, automatically recreating or replacing them if a node fails or a pod terminates, thus providing a self‑healing, scalable deployment mechanism.  

[Source: Concepts.pdf, Page: 5]  
[Source: Tutorials.pdf, Page: 9]  
[Source: Tutorials.pdf, Page: 2]
############################################################
Query: What is a Kubernetes Pod?
Response: A Kubernetes Pod is the smallest deployable unit in a Kubernetes cluster. It is a logical host that groups one or more containers (often just one) along with shared resources such as storage volumes, a unique cluster IP address, and runtime information (image version, ports, etc.). All containers in a Pod are co‑located, co‑scheduled, and share Linux namespaces and cgroups, providing a sha

#### ***negative/unsupported queries***

In [71]:
unrelated_queries = [
    "What is the capital of France?",
    "How does a Python decorator work?",
    "What is machine learning?",
    "How does a SQL JOIN work?",
    "What is the difference between TCP and UDP?",
]
for query in unrelated_queries:
    print("Query:",query)
    response = final_rag_response(query)
    print("Response:",response)
    print("#"*60)

Query: What is the capital of France?
Response: I don't have information based on the provided documents.
############################################################
Query: How does a Python decorator work?
Response: I don't have information based on the provided documents.
############################################################
Query: What is machine learning?
Response: I don't have information based on the provided documents.
############################################################
Query: How does a SQL JOIN work?
Response: I don't have information based on the provided documents.
############################################################
Query: What is the difference between TCP and UDP?
Response: TCP is a connection‑oriented, reliable transport and is the default protocol for services in Kubernetes.  
UDP is a connection‑less, best‑effort transport; it is used for most services but its support in load balancers depends on the cloud provider. In practice, UDP connections

#### **Grounding / Unsupported-Question Evaluation.**

In [72]:
grounding_queries = [
    # Supported by the corpus
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "How does a Deployment manage ReplicaSets and Pods during an update?",

    # Clearly unsupported
    "What is the capital of France?",
    "How does a Python decorator work?",
    "What is a SQL JOIN?",
    "What is machine learning?",
    "How does TCP congestion control work?",
]
for query in grounding_queries:
    print("Query:",query)
    response = final_rag_response(query)
    print("Response:",response)
    print("#"*60)

Query: What is a Kubernetes Deployment?
Response: A Kubernetes Deployment is an object that declares the desired state of an application—typically the number of replica Pods and the container image to run. The Deployment controller creates, updates, and continuously monitors those Pods, automatically recreating any that fail or are removed, thus providing self‑healing and scaling for the application.  

[Source: Tutorials.pdf, Page: 9]  
[Source: Concepts.pdf, Page: 5]  
[Source: Tutorials.pdf, Page: 2]
############################################################
Query: What is a Kubernetes Pod?
Response: A Kubernetes Pod is the smallest deployable unit in Kubernetes. It is a logical host that groups one or more containers which share the same storage volumes, network (a single cluster IP), and a set of Linux namespaces and cgroups. All containers in a pod are co‑located and co‑scheduled on the same node, and the pod provides a shared context for running the containers together.  

[So

#### **Final RAG quality check.**

In [73]:
final_rag_queries = [
    "What happens when a Deployment specifies three replicas?",
    "What happens if one of the Pods managed by a Deployment fails?",
    "How does a Service identify which Pods should receive traffic?",
    "How do containers within the same Pod communicate with each other?",
    "What is the difference between a Pod and a Deployment?",
    "What is the difference between a Deployment and a ReplicaSet?",
    "How are Deployments, ReplicaSets, and Pods related?",
    "How does a Deployment use ReplicaSets to manage Pods?",
    "How does a Service provide access to Pods managed by a Deployment?",
    "How can multiple copies of the same application run in Kubernetes?",
]
for query in final_rag_queries:
    print("Query:",query)
    response = final_rag_response(query)
    print("Response:",response)
    print("#"*60)

Query: What happens when a Deployment specifies three replicas?
Response: When a Deployment declares `replicas: 3`, the Kubernetes control plane attempts to run three instances of the application. It creates a ReplicaSet, scales it up to ensure that three Pods are available, and may temporarily create up to four Pods (the desired 3 + maxSurge) during a rolling update. If a ResourceQuota or other restriction prevents it, fewer Pods may be created and the status will show the actual number available.  

[Source: Concepts.pdf, Page: 5]  
[Source: Tasks.pdf, Page: 103]  
[Source: Concepts.pdf, Page: 137]
############################################################
Query: What happens if one of the Pods managed by a Deployment fails?
Response: If a Pod that is owned by a Deployment crashes or is otherwise removed, the Deployment’s ReplicaSet automatically detects the loss and creates a new Pod to replace it. The new Pod is then scheduled onto a healthy node by the Kubernetes scheduler.  

[